# 50. Command-line Apps — `click · typer`
**Python Libraries Lab · Phase 3 · Utilities & Testing**

_Build friendly terminal tools with options, help and subcommands._

⚙️ **Runs in Colab** after the setup cell below.

Many useful tools are run from a terminal, like git or pip. Click and Typer make it easy to build your own. You write normal Python functions, and they turn them into commands with options, arguments, automatic help text, and helpful error messages when someone types something wrong.

**In this notebook:** You will build a ShopEasy admin tool with Click that applies discounts, then a Typer version with subcommands to add and list products. You will run both from the terminal and test them automatically with a CLI runner.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q click typer
import importlib.metadata as md
for pkg in ['click', 'typer']:
    print(pkg, md.version(pkg))

## Step 1 · A Click command
Each decorator adds an argument or option. Help text comes from the docstring and `help=`.

In [ ]:
%%writefile discount_click.py
import click

@click.command()
@click.argument("amount", type=float)
@click.option("--percent", "-p", default=10.0, show_default=True, help="Discount percent.")
@click.option("--gst/--no-gst", default=True, help="Add 18% GST after the discount.")
def discount(amount, percent, gst):
    """Apply a ShopEasy discount to AMOUNT."""
    if not 0 <= percent <= 90:
        raise click.BadParameter("percent must be between 0 and 90", param_hint="--percent")
    final = amount * (1 - percent / 100) * (1.18 if gst else 1)
    click.echo(f"Pay Rs.{final:,.2f}")

if __name__ == "__main__":
    discount()

## Step 2 · Run it like a user
Help, normal use, a flag, and a helpful error.

In [ ]:
!python discount_click.py --help
!python discount_click.py 1000 -p 20
!python discount_click.py 1000 --percent 20 --no-gst
!python discount_click.py 1000 --percent 95

## Step 3 · A Typer app with subcommands
Type hints become options. Each function becomes a subcommand.

In [ ]:
%%writefile shopcli.py
import json
from pathlib import Path
import typer

app = typer.Typer(help="Manage the ShopEasy catalogue.")
DB = Path("catalogue.json")

def load():
    return json.loads(DB.read_text()) if DB.exists() else []

@app.command()
def add(name: str, price: float = typer.Option(..., min=1, help="Price in rupees"), stock: int = 0):
    """Add a product."""
    items = load()
    items.append({"name": name, "price": price, "stock": stock})
    DB.write_text(json.dumps(items, indent=2))
    typer.echo(f"Added {name}")

@app.command("list")
def list_products(max_price: float = typer.Option(None, help="Only show cheaper items")):
    """List products."""
    for p in load():
        if max_price is None or p["price"] <= max_price:
            typer.echo(f"{p['name']:<14} Rs.{p['price']:>8,.0f}  stock {p['stock']}")

if __name__ == "__main__":
    app()

## Step 4 · Use the Typer app
Subcommands, automatic validation and help.

In [ ]:
!rm -f catalogue.json
!python shopcli.py --help
!python shopcli.py add "Steel Bottle" --price 499 --stock 40
!python shopcli.py add "Yoga Mat" --price 1299
!python shopcli.py add "Bad Item" --price 0
!python shopcli.py list --max-price 1000

## Step 5 · Test command-line apps
`CliRunner` runs commands inside Python and captures output and exit codes.

In [ ]:
from click.testing import CliRunner
from typer.testing import CliRunner as TyperRunner
import importlib, discount_click, shopcli
importlib.reload(discount_click); importlib.reload(shopcli)

res = CliRunner().invoke(discount_click.discount, ["2000", "-p", "50", "--no-gst"])
print(res.exit_code, res.output.strip())
res = CliRunner().invoke(discount_click.discount, ["2000", "-p", "99"])
print(res.exit_code, res.output.strip().splitlines()[-1])
res = TyperRunner().invoke(shopcli.app, ["list"])
print(res.exit_code, res.output)

## ✍️ Your turn
Use `CliRunner` to run `discount_click.discount` with `['500', '--percent', '10', '--no-gst']` and store the **output text, stripped** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == "Pay Rs.450.00", f"Expected 'Pay Rs.450.00', got {answer!r}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = CliRunner().invoke(discount_click.discount, ["500", "--percent", "10", "--no-gst"]).output.strip()

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Command**
- **Option / argument**
- **--help**
- **Subcommand**
- **CliRunner**
- **Exit code**

**Next:** 51. Rich Terminal Output (`rich`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/51_rich.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install click typer
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.